# svm_v3_full_history_hyperparameter_test

SVM hyperparameter test using the full-history session-aligned dataset and the same simple feature groups as `notebooksv2/svm_v1.ipynb` / `notebooksv3/svm_v2.ipynb`.

The notebook tests a small SVM grid on the validation split, selects the best parameter set per feature set, and evaluates those selected configurations on the test split.

In [21]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)

In [22]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = {
    "dataset_path": PROJECT_ROOT / "data" / "datasets" / "stock_panel_nine_tickers_session_aligned_full_history_raw.csv",
    "excluded_tickers": ["NFLX"],
    "neutral_band": 0.005,
    "test_size": 0.20,
    "validation_fraction_within_pretest": 0.15,
    "min_validation_dates": 20,
    "gap_days": 1,
    "random_state": 42,
}

SVM_PARAM_GRID = [
    {"param_set": "linear_C0.1_balanced", "kernel": "linear", "C": 0.1, "class_weight": "balanced"},
    {"param_set": "linear_C1_balanced", "kernel": "linear", "C": 1.0, "class_weight": "balanced"},
    {"param_set": "linear_C10_balanced", "kernel": "linear", "C": 10.0, "class_weight": "balanced"},
    {"param_set": "rbf_C0.1_scale_balanced", "kernel": "rbf", "C": 0.1, "gamma": "scale", "class_weight": "balanced"},
    {"param_set": "rbf_C1_scale_balanced", "kernel": "rbf", "C": 1.0, "gamma": "scale", "class_weight": "balanced"},
    {"param_set": "rbf_C10_scale_balanced", "kernel": "rbf", "C": 10.0, "gamma": "scale", "class_weight": "balanced"},
    {"param_set": "rbf_C1_gamma0.01_balanced", "kernel": "rbf", "C": 1.0, "gamma": 0.01, "class_weight": "balanced"},
    {"param_set": "rbf_C1_gamma0.1_balanced", "kernel": "rbf", "C": 1.0, "gamma": 0.1, "class_weight": "balanced"},
]

GOOGLE_TRENDS_FEATURES = [
    # "google_trends_zscore_90d",
    "google_trends_above_ticker_train_median",
]
DERIVED_FEATURE_COLUMNS = {"google_trends_above_ticker_train_median"}

FEATURE_SETS = {
    "Model A - price only": [
        "return_1d",
        "return_5d",
        "return_20d",
        "rolling_volatility_20d",
    ],
    "Model B - price + volume": [
        "return_1d",
        "return_5d",
        "return_20d",
        "rolling_volatility_20d",
        "volume_zscore_20d",
    ],
    "Model C - price + volume + GDELT": [
        "return_1d",
        "return_5d",
        "return_20d",
        "rolling_volatility_20d",
        "volume_zscore_20d",
        "gdelt_sentiment_zscore_30d",
        "gdelt_article_count_zscore_30d",
    ],
    "Model D - price + volume + GDELT + Reddit": [
        "return_1d",
        "return_5d",
        "return_20d",
        "rolling_volatility_20d",
        "volume_zscore_20d",
        "gdelt_sentiment_zscore_30d",
        "gdelt_article_count_zscore_30d",
        "reddit_sentiment_zscore_30d",
        "reddit_comment_count_zscore_30d",
    ],
    "Model E - price + volume + Reddit": [
        "return_1d",
        "return_5d",
        "return_20d",
        "rolling_volatility_20d",
        "volume_zscore_20d",
        "reddit_sentiment_zscore_30d",
        "reddit_comment_count_zscore_30d",
    ],
    "Model F - price + volume + all alternative data": [
        "return_1d",
        "return_5d",
        "return_20d",
        "rolling_volatility_20d",
        "volume_zscore_20d",
        "gdelt_sentiment_zscore_30d",
        "gdelt_article_count_zscore_30d",
        "reddit_sentiment_zscore_30d",
        "reddit_comment_count_zscore_30d",
        *GOOGLE_TRENDS_FEATURES,
    ],
    "Model G - price + volume + google": [
        "return_1d",
        "return_5d",
        "return_20d",
        "rolling_volatility_20d",
        "volume_zscore_20d",
        *GOOGLE_TRENDS_FEATURES,
    ],
}

FEATURE_SETS_TO_TEST = list(FEATURE_SETS.keys())
BASELINE_FEATURE_SET = "Model B - price + volume"

pd.DataFrame(SVM_PARAM_GRID)

,param_set,kernel,C,class_weight,gamma
0,linear_C0.1_balanced,linear,0.1,balanced,NaN
1,linear_C1_balanced,linear,1.0,balanced,NaN
2,linear_C10_balanced,linear,10.0,balanced,NaN
3,rbf_C0.1_scale_balanced,rbf,0.1,balanced,scale
4,rbf_C1_scale_balanced,rbf,1.0,balanced,scale
5,rbf_C10_scale_balanced,rbf,10.0,balanced,scale
6,rbf_C1_gamma0.01_balanced,rbf,1.0,balanced,0.01
7,rbf_C1_gamma0.1_balanced,rbf,1.0,balanced,0.1


In [23]:
def add_trailing_zscore(frame: pd.DataFrame, source_col: str, output_col: str, window: int) -> None:
    grouped = frame.groupby("ticker")[source_col]
    rolling_mean = grouped.transform(lambda s: s.shift(1).rolling(window).mean())
    rolling_std = grouped.transform(lambda s: s.shift(1).rolling(window).std())
    frame[output_col] = (frame[source_col] - rolling_mean) / rolling_std.replace(0.0, np.nan)


def build_feature_frame(raw_df: pd.DataFrame, neutral_band: float) -> pd.DataFrame:
    frame = raw_df.copy().sort_values(["ticker", "date"]).reset_index(drop=True)
    frame["comm_reddit_vader_mean"] = frame["comm_reddit_vader_mean"].fillna(0.0)
    frame["comm_reddit_posts"] = frame["comm_reddit_posts"].fillna(0.0)

    price_group = frame.groupby("ticker")["stock_price"]
    frame["return_1d"] = price_group.pct_change(1)
    frame["return_5d"] = price_group.pct_change(5)
    frame["return_20d"] = price_group.pct_change(20)
    frame["rolling_volatility_20d"] = (
        frame.groupby("ticker")["return_1d"].transform(lambda s: s.shift(1).rolling(20).std())
    )

    add_trailing_zscore(frame, "stock_volume", "volume_zscore_20d", 10)
    add_trailing_zscore(frame, "gdelt_sentiment_score", "gdelt_sentiment_zscore_30d", 10)
    add_trailing_zscore(frame, "gdelt_articles", "gdelt_article_count_zscore_30d", 6)
    add_trailing_zscore(frame, "comm_reddit_vader_mean", "reddit_sentiment_zscore_30d", 6)
    add_trailing_zscore(frame, "comm_reddit_posts", "reddit_comment_count_zscore_30d", 6)
    add_trailing_zscore(frame, "google_trends_score", "google_trends_zscore_90d", 30)

    frame["future_return_1d"] = price_group.shift(-1) / frame["stock_price"] - 1.0
    frame["target"] = np.select(
        [
            frame["future_return_1d"] < -neutral_band,
            frame["future_return_1d"] > neutral_band,
        ],
        [0, 1],
        default=np.nan,
    )
    frame["target_available"] = frame["future_return_1d"].notna()
    frame["is_neutral"] = frame["target_available"] & frame["future_return_1d"].abs().le(neutral_band)
    return frame


def make_train_validation_test_split_by_date(
    frame: pd.DataFrame,
    test_size: float,
    validation_fraction_within_pretest: float,
    min_validation_dates: int,
    gap_days: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    unique_dates = sorted(frame["date"].drop_duplicates())

    test_start_idx = int(np.floor(len(unique_dates) * (1.0 - test_size)))
    test_start_idx = min(max(test_start_idx, 2), len(unique_dates) - 1)
    pretest_end_idx = max(test_start_idx - gap_days, 1)
    pretest_dates = unique_dates[:pretest_end_idx]

    validation_size = int(np.floor(len(pretest_dates) * validation_fraction_within_pretest))
    validation_size = max(min_validation_dates, validation_size)
    validation_size = min(max(validation_size, 1), len(pretest_dates) - 1)

    validation_start_idx = len(pretest_dates) - validation_size
    train_end_idx = max(validation_start_idx - gap_days, 1)

    train_dates = unique_dates[:train_end_idx]
    validation_dates = pretest_dates[validation_start_idx:]
    test_dates = unique_dates[test_start_idx:]

    train_df = frame[frame["date"].isin(train_dates)].copy()
    validation_df = frame[frame["date"].isin(validation_dates)].copy()
    test_df = frame[frame["date"].isin(test_dates)].copy()
    return train_df, validation_df, test_df


def safe_auc(y_true: pd.Series, scores: np.ndarray) -> float:
    if y_true.nunique() < 2:
        return np.nan
    return float(roc_auc_score(y_true, scores))


def build_svm_pipeline_from_params(params: dict) -> Pipeline:
    svm_params = dict(params)
    svm_params.pop("param_set", None)
    if svm_params.get("kernel") == "linear":
        svm_params.pop("gamma", None)
    svm_params.setdefault("class_weight", "balanced")
    svm_params.setdefault("random_state", CONFIG["random_state"])
    svm_params["probability"] = False
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(**svm_params)),
        ]
    )

def add_google_trends_train_median_feature(
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    feature_name = "google_trends_above_ticker_train_median"
    source_col = "google_trends_score"
    median_by_ticker = train_input_df.groupby("ticker")[source_col].median()
    fallback_median = train_input_df[source_col].median()

    def with_feature(frame: pd.DataFrame) -> pd.DataFrame:
        out = frame.copy()
        thresholds = out["ticker"].map(median_by_ticker)
        if pd.notna(fallback_median):
            thresholds = thresholds.fillna(fallback_median)
        out[feature_name] = np.where(
            out[source_col].notna() & thresholds.notna(),
            (out[source_col] > thresholds).astype(float),
            np.nan,
        )
        return out

    return with_feature(train_input_df), with_feature(eval_input_df)

In [24]:
raw_df = pd.read_csv(CONFIG["dataset_path"], parse_dates=["date"])
raw_df = raw_df[~raw_df["ticker"].isin(CONFIG["excluded_tickers"])].copy()
raw_df = raw_df.sort_values(["ticker", "date"]).reset_index(drop=True)

feature_df = build_feature_frame(raw_df, neutral_band=CONFIG["neutral_band"])
modeled_df = feature_df[feature_df["target"].isin([0.0, 1.0])].copy()
modeled_df["target"] = modeled_df["target"].astype(int)

train_df, validation_df, test_df = make_train_validation_test_split_by_date(
    modeled_df,
    test_size=CONFIG["test_size"],
    validation_fraction_within_pretest=CONFIG["validation_fraction_within_pretest"],
    min_validation_dates=CONFIG["min_validation_dates"],
    gap_days=CONFIG["gap_days"],
)

split_summary_df = pd.DataFrame(
    [
        {"split": "train", "n_rows": len(train_df), "n_dates": train_df["date"].nunique(), "date_min": train_df["date"].min(), "date_max": train_df["date"].max()},
        {"split": "validation", "n_rows": len(validation_df), "n_dates": validation_df["date"].nunique(), "date_min": validation_df["date"].min(), "date_max": validation_df["date"].max()},
        {"split": "test", "n_rows": len(test_df), "n_dates": test_df["date"].nunique(), "date_min": test_df["date"].min(), "date_max": test_df["date"].max()},
    ]
)

split_summary_df

,split,n_rows,n_dates,date_min,date_max
0,train,5351,851,2021-01-04,2024-05-21
1,validation,902,150,2024-05-23,2024-12-26
2,test,1513,251,2024-12-30,2025-12-30


In [25]:
missing_requested_feature_sets = [name for name in FEATURE_SETS_TO_TEST if name not in FEATURE_SETS]
if missing_requested_feature_sets:
    raise KeyError(f"Unknown feature sets: {missing_requested_feature_sets}")

missing_feature_columns = sorted(
    {
        feature
        for name in FEATURE_SETS_TO_TEST
        for feature in FEATURE_SETS[name]
        if feature not in feature_df.columns and feature not in DERIVED_FEATURE_COLUMNS
    }
)
if missing_feature_columns:
    raise KeyError(f"Missing feature columns: {missing_feature_columns}")

print(f"Feature sets to test: {len(FEATURE_SETS_TO_TEST)}")
print(f"SVM parameter sets: {len(SVM_PARAM_GRID)}")
print(f"Validation fits: {len(FEATURE_SETS_TO_TEST) * len(SVM_PARAM_GRID)}")

Feature sets to test: 7
SVM parameter sets: 8
Validation fits: 56


In [26]:
def evaluate_svm_params(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
    split_name: str,
) -> dict:
    train_features_df = train_input_df
    eval_features_df = eval_input_df
    if "google_trends_above_ticker_train_median" in features:
        train_features_df, eval_features_df = add_google_trends_train_median_feature(
            train_input_df,
            eval_input_df,
        )

    pipeline = build_svm_pipeline_from_params(params)
    pipeline.fit(train_features_df[features], train_features_df["target"])
    preds = pipeline.predict(eval_features_df[features])
    scores = pipeline.decision_function(eval_features_df[features])
    return {
        "split": split_name,
        "feature_set": feature_set_name,
        "param_set": params["param_set"],
        "kernel": params.get("kernel"),
        "C": params.get("C"),
        "gamma": params.get("gamma", None),
        "class_weight": params.get("class_weight"),
        "n_features": len(features),
        "train_rows": len(train_input_df),
        "eval_rows": len(eval_input_df),
        "eval_positive_rate": float(eval_features_df["target"].mean()),
        "predicted_positive_rate": float(np.mean(preds)),
        "accuracy": float(accuracy_score(eval_features_df["target"], preds)),
        "balanced_accuracy": float(balanced_accuracy_score(eval_features_df["target"], preds)),
        "auc": safe_auc(eval_features_df["target"], scores),
        "n_support_vectors": int(pipeline.named_steps["model"].n_support_.sum()),
    }

In [27]:
validation_grid_rows = []

for feature_set_name in FEATURE_SETS_TO_TEST:
    features = FEATURE_SETS[feature_set_name]
    for params in SVM_PARAM_GRID:
        validation_grid_rows.append(
            evaluate_svm_params(
                feature_set_name=feature_set_name,
                features=features,
                params=params,
                train_input_df=train_df,
                eval_input_df=validation_df,
                split_name="validation",
            )
        )

validation_grid_results_df = pd.DataFrame(validation_grid_rows).sort_values(
    ["balanced_accuracy", "accuracy", "auc", "feature_set", "param_set"],
    ascending=[False, False, False, True, True],
).reset_index(drop=True)

validation_grid_results_df

,split,feature_set,param_set,kernel,C,gamma,class_weight,n_features,train_rows,eval_rows,eval_positive_rate,predicted_positive_rate,accuracy,balanced_accuracy,auc,n_support_vectors
0,validation,Model B - price + volume,rbf_C1_scale_balanced,rbf,1.0,scale,balanced,5,5351,902,0.559867,0.596452,0.537694,0.526526,0.525625,5118
1,validation,Model B - price + volume,rbf_C10_scale_balanced,rbf,10.0,scale,balanced,5,5351,902,0.559867,0.553215,0.532151,0.526154,0.530394,4948
2,validation,Model A - price only,rbf_C10_scale_balanced,rbf,10.0,scale,balanced,4,5351,902,0.559867,0.544346,0.525499,0.520483,0.532554,4976
3,validation,Model A - price only,rbf_C1_scale_balanced,rbf,1.0,scale,balanced,4,5351,902,0.559867,0.620843,0.533259,0.519064,0.525381,5109
4,validation,Model B - price + volume,rbf_C1_gamma0.01_balanced,rbf,1.0,0.01,balanced,5,5351,902,0.559867,0.675166,0.538803,0.518089,0.540529,5245
5,validation,Model A - price only,rbf_C1_gamma0.01_balanced,rbf,1.0,0.01,balanced,4,5351,902,0.559867,0.671840,0.537694,0.517368,0.544300,5246
6,validation,Model C - price + volume + GDELT,linear_C0.1_balanced,linear,0.1,None,balanced,7,5351,902,0.559867,0.771619,0.548780,0.516495,0.510123,5209
7,validation,Model B - price + volume,rbf_C0.1_scale_balanced,rbf,0.1,scale,balanced,5,5351,902,0.559867,0.618625,0.528825,0.514834,0.532324,5239
8,validation,Model C - price + volume + GDELT,linear_C10_balanced,linear,10.0,None,balanced,7,5351,902,0.559867,0.769401,0.546563,0.514515,0.509415,5207
9,validation,Model C - price + volume + GDELT,linear_C1_balanced,linear,1.0,None,balanced,7,5351,902,0.559867,0.769401,0.546563,0.514515,0.509415,5207


In [28]:
validation_accuracy_by_feature_set_df = (
    validation_grid_results_df.sort_values(
        ["feature_set", "accuracy", "balanced_accuracy", "auc"],
        ascending=[True, False, False, False],
    )
    .groupby("feature_set", as_index=False)
    .head(1)
    .rename(
        columns={
            "param_set": "best_param_set_by_accuracy",
            "accuracy": "validation_accuracy",
            "balanced_accuracy": "validation_balanced_accuracy",
            "auc": "validation_auc",
        }
    )
    .sort_values(
        ["validation_accuracy", "validation_balanced_accuracy", "validation_auc"],
        ascending=False,
    )
    .reset_index(drop=True)
)

validation_accuracy_by_feature_set_df[
    [
        "feature_set",
        "best_param_set_by_accuracy",
        "kernel",
        "C",
        "gamma",
        "validation_accuracy",
        "validation_balanced_accuracy",
        "validation_auc",
        "predicted_positive_rate",
    ]
]

,feature_set,best_param_set_by_accuracy,kernel,C,gamma,validation_accuracy,validation_balanced_accuracy,validation_auc,predicted_positive_rate
0,Model C - price + volume + GDELT,linear_C0.1_balanced,linear,0.1,None,0.548780,0.516495,0.510123,0.771619
1,Model D - price + volume + GDELT + Reddit,linear_C10_balanced,linear,10.0,None,0.542129,0.512170,0.508257,0.751663
2,Model E - price + volume + Reddit,linear_C0.1_balanced,linear,0.1,None,0.541020,0.508756,0.531661,0.770510
3,Model B - price + volume,rbf_C1_gamma0.01_balanced,rbf,1.0,0.01,0.538803,0.518089,0.540529,0.675166
4,Model A - price only,rbf_C1_gamma0.01_balanced,rbf,1.0,0.01,0.537694,0.517368,0.544300,0.671840
5,Model F - price + volume + all alternative data,linear_C10_balanced,linear,10.0,None,0.502217,0.495381,0.498830,0.556541
6,Model G - price + volume + google,rbf_C1_gamma0.01_balanced,rbf,1.0,0.01,0.494457,0.509998,0.514183,0.371397


In [29]:
best_validation_params_df = (
    validation_grid_results_df.sort_values(
        ["feature_set", "balanced_accuracy", "accuracy", "auc"],
        ascending=[True, False, False, False],
    )
    .groupby("feature_set", as_index=False)
    .head(1)
    .sort_values(["balanced_accuracy", "accuracy", "auc"], ascending=False)
    .reset_index(drop=True)
)

best_validation_params_df

,split,feature_set,param_set,kernel,C,gamma,class_weight,n_features,train_rows,eval_rows,eval_positive_rate,predicted_positive_rate,accuracy,balanced_accuracy,auc,n_support_vectors
0,validation,Model B - price + volume,rbf_C1_scale_balanced,rbf,1.0,scale,balanced,5,5351,902,0.559867,0.596452,0.537694,0.526526,0.525625,5118
1,validation,Model A - price only,rbf_C10_scale_balanced,rbf,10.0,scale,balanced,4,5351,902,0.559867,0.544346,0.525499,0.520483,0.532554,4976
2,validation,Model C - price + volume + GDELT,linear_C0.1_balanced,linear,0.1,None,balanced,7,5351,902,0.559867,0.771619,0.548780,0.516495,0.510123,5209
3,validation,Model D - price + volume + GDELT + Reddit,linear_C10_balanced,linear,10.0,None,balanced,9,5351,902,0.559867,0.751663,0.542129,0.512170,0.508257,5206
4,validation,Model G - price + volume + google,rbf_C1_gamma0.01_balanced,rbf,1.0,0.01,balanced,6,5351,902,0.559867,0.371397,0.494457,0.509998,0.514183,5225
5,validation,Model E - price + volume + Reddit,rbf_C1_gamma0.1_balanced,rbf,1.0,0.1,balanced,7,5351,902,0.559867,0.589800,0.519956,0.509337,0.506921,5140
6,validation,Model F - price + volume + all alternative data,rbf_C1_gamma0.01_balanced,rbf,1.0,0.01,balanced,10,5351,902,0.559867,0.461197,0.493348,0.497965,0.499219,5188


In [30]:
test_rows = []
train_validation_df = pd.concat([train_df, validation_df], axis=0).reset_index(drop=True)

for row in best_validation_params_df.to_dict(orient="records"):
    params = {
        "param_set": row["param_set"],
        "kernel": row["kernel"],
        "C": row["C"],
        "gamma": row["gamma"],
        "class_weight": row["class_weight"],
    }
    if pd.isna(params["gamma"]):
        params.pop("gamma")
    feature_set_name = row["feature_set"]
    test_rows.append(
        evaluate_svm_params(
            feature_set_name=feature_set_name,
            features=FEATURE_SETS[feature_set_name],
            params=params,
            train_input_df=train_validation_df,
            eval_input_df=test_df,
            split_name="test_best_validation_params",
        )
    )

test_best_validation_params_df = pd.DataFrame(test_rows).sort_values(
    ["balanced_accuracy", "accuracy", "auc", "feature_set"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

test_best_validation_params_df

,split,feature_set,param_set,kernel,C,gamma,class_weight,n_features,train_rows,eval_rows,eval_positive_rate,predicted_positive_rate,accuracy,balanced_accuracy,auc,n_support_vectors
0,test_best_validation_params,Model A - price only,rbf_C10_scale_balanced,rbf,10.0,scale,balanced,4,6253,1513,0.521481,0.537343,0.539987,0.538453,0.547226,5818
1,test_best_validation_params,Model E - price + volume + Reddit,rbf_C1_gamma0.1_balanced,rbf,1.0,0.1,balanced,7,6253,1513,0.521481,0.524124,0.532056,0.531076,0.541156,6006
2,test_best_validation_params,Model F - price + volume + all alternative data,rbf_C1_gamma0.01_balanced,rbf,1.0,0.01,balanced,10,6253,1513,0.521481,0.506279,0.523463,0.523236,0.525604,6079
3,test_best_validation_params,Model B - price + volume,rbf_C1_scale_balanced,rbf,1.0,scale,balanced,5,6253,1513,0.521481,0.535360,0.523463,0.521985,0.525382,5975
4,test_best_validation_params,Model G - price + volume + google,rbf_C1_gamma0.01_balanced,rbf,1.0,0.01,balanced,6,6253,1513,0.521481,0.561798,0.511566,0.508928,0.513198,6114
5,test_best_validation_params,Model D - price + volume + GDELT + Reddit,linear_C10_balanced,linear,10.0,None,balanced,9,6253,1513,0.521481,0.765367,0.519498,0.508112,0.523146,6106
6,test_best_validation_params,Model C - price + volume + GDELT,linear_C0.1_balanced,linear,0.1,None,balanced,7,6253,1513,0.521481,0.773959,0.516193,0.504432,0.520142,6108


In [31]:
baseline_validation_row = best_validation_params_df[best_validation_params_df["feature_set"].eq(BASELINE_FEATURE_SET)].iloc[0]
baseline_test_row = test_best_validation_params_df[test_best_validation_params_df["feature_set"].eq(BASELINE_FEATURE_SET)].iloc[0]

simple_hyperparameter_summary_df = best_validation_params_df.merge(
    test_best_validation_params_df[
        ["feature_set", "accuracy", "balanced_accuracy", "auc", "predicted_positive_rate"]
    ].rename(
        columns={
            "accuracy": "test_accuracy",
            "balanced_accuracy": "test_balanced_accuracy",
            "auc": "test_auc",
            "predicted_positive_rate": "test_predicted_positive_rate",
        }
    ),
    on="feature_set",
    how="left",
)

simple_hyperparameter_summary_df = simple_hyperparameter_summary_df.rename(
    columns={
        "accuracy": "validation_accuracy",
        "balanced_accuracy": "validation_balanced_accuracy",
        "auc": "validation_auc",
        "predicted_positive_rate": "validation_predicted_positive_rate",
    }
)

simple_hyperparameter_summary_df["validation_accuracy_lift_vs_baseline"] = (
    simple_hyperparameter_summary_df["validation_accuracy"] - baseline_validation_row["accuracy"]
)
simple_hyperparameter_summary_df["validation_balanced_accuracy_lift_vs_baseline"] = (
    simple_hyperparameter_summary_df["validation_balanced_accuracy"] - baseline_validation_row["balanced_accuracy"]
)
simple_hyperparameter_summary_df["validation_auc_lift_vs_baseline"] = (
    simple_hyperparameter_summary_df["validation_auc"] - baseline_validation_row["auc"]
)
simple_hyperparameter_summary_df["test_accuracy_lift_vs_baseline"] = (
    simple_hyperparameter_summary_df["test_accuracy"] - baseline_test_row["accuracy"]
)
simple_hyperparameter_summary_df["test_balanced_accuracy_lift_vs_baseline"] = (
    simple_hyperparameter_summary_df["test_balanced_accuracy"] - baseline_test_row["balanced_accuracy"]
)
simple_hyperparameter_summary_df["test_auc_lift_vs_baseline"] = (
    simple_hyperparameter_summary_df["test_auc"] - baseline_test_row["auc"]
)

summary_columns = [
    "feature_set",
    "param_set",
    "kernel",
    "C",
    "gamma",
    "n_features",
    "validation_accuracy",
    "validation_balanced_accuracy",
    "validation_auc",
    "validation_accuracy_lift_vs_baseline",
    "validation_balanced_accuracy_lift_vs_baseline",
    "validation_auc_lift_vs_baseline",
    "test_accuracy",
    "test_balanced_accuracy",
    "test_auc",
    "test_accuracy_lift_vs_baseline",
    "test_balanced_accuracy_lift_vs_baseline",
    "test_auc_lift_vs_baseline",
]

simple_hyperparameter_summary_df[summary_columns]

,feature_set,param_set,kernel,C,gamma,n_features,validation_accuracy,validation_balanced_accuracy,validation_auc,validation_accuracy_lift_vs_baseline,validation_balanced_accuracy_lift_vs_baseline,validation_auc_lift_vs_baseline,test_accuracy,test_balanced_accuracy,test_auc,test_accuracy_lift_vs_baseline,test_balanced_accuracy_lift_vs_baseline,test_auc_lift_vs_baseline
0,Model B - price + volume,rbf_C1_scale_balanced,rbf,1.0,scale,5,0.537694,0.526526,0.525625,0.000000,0.000000,0.000000,0.523463,0.521985,0.525382,0.000000,0.000000,0.000000
1,Model A - price only,rbf_C10_scale_balanced,rbf,10.0,scale,4,0.525499,0.520483,0.532554,-0.012195,-0.006043,0.006928,0.539987,0.538453,0.547226,0.016523,0.016469,0.021844
2,Model C - price + volume + GDELT,linear_C0.1_balanced,linear,0.1,None,7,0.548780,0.516495,0.510123,0.011086,-0.010031,-0.015502,0.516193,0.504432,0.520142,-0.007270,-0.017553,-0.005240
3,Model D - price + volume + GDELT + Reddit,linear_C10_balanced,linear,10.0,None,9,0.542129,0.512170,0.508257,0.004435,-0.014355,-0.017368,0.519498,0.508112,0.523146,-0.003966,-0.013873,-0.002236
4,Model G - price + volume + google,rbf_C1_gamma0.01_balanced,rbf,1.0,0.01,6,0.494457,0.509998,0.514183,-0.043237,-0.016527,-0.011442,0.511566,0.508928,0.513198,-0.011897,-0.013057,-0.012184
5,Model E - price + volume + Reddit,rbf_C1_gamma0.1_balanced,rbf,1.0,0.1,7,0.519956,0.509337,0.506921,-0.017738,-0.017188,-0.018705,0.532056,0.531076,0.541156,0.008592,0.009092,0.015775
6,Model F - price + volume + all alternative data,rbf_C1_gamma0.01_balanced,rbf,1.0,0.01,10,0.493348,0.497965,0.499219,-0.044346,-0.028561,-0.026406,0.523463,0.523236,0.525604,0.000000,0.001252,0.000222
